# Prediction - Eğitilmiş Model ile Tahmin Yapma

Bu notebook, eğitilmiş custom NER modelini kullanarak yeni metinler üzerinde tahmin yapar.

## Adımlar:
1. **Setup & License** - Spark NLP Healthcare lisansı ve ortam kurulumu
2. **Model Yükleme** - Eğitilmiş custom NER modelini yükleme
3. **Tahmin Yapma** - Yeni metinler üzerinde entity tahminleri
4. **Sonuçları Görselleştirme** - Tahmin sonuçlarını görselleştirme

**Gereksinimler:**
- `2_model_training.ipynb` notebook'unun çalıştırılmış olması
- `models/trained/custom_ner_model` modelinin mevcut olması


## 1. Setup & License Configuration


In [ ]:
import json
import os
from pathlib import Path

# License dosyasını yükle
try:
    with open('/kaggle/input/spark-jsl/spark_jsl.json') as f:
        license_keys = json.load(f)
except:
    try:
        from google.colab import files
        if 'spark_jsl.json' not in os.listdir():
            print("Please upload your spark_jsl.json license file:")
            uploaded = files.upload()
            os.rename(list(uploaded.keys())[0], 'spark_jsl.json')
        with open('spark_jsl.json') as f:
            license_keys = json.load(f)
    except:
        with open('spark_jsl.json') as f:
            license_keys = json.load(f)

locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")


In [ ]:
# Kütüphaneleri yükle
import subprocess

try:
    import torch
    gpu_available = torch.cuda.is_available()
except:
    gpu_available = False

if not gpu_available:
    %pip install -q torch torchvision torchaudio

%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET
%pip install -q spark-nlp-display pandas numpy

print("✅ Libraries installed")


In [ ]:
# Spark Session başlat
import sparknlp
import sparknlp_jsl
from pyspark.sql import SparkSession

try:
    import torch
    gpu_available = torch.cuda.is_available()
except:
    gpu_available = False

params = {
    "spark.driver.memory": "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": "/content/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": "/content/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled")

spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
print(f"✅ Spark NLP Version: {sparknlp.version()}")
print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
print("✅ Spark session initialized")


In [ ]:
# Modülleri import et
import sys
import os
from pathlib import Path

# Notebook'un bulunduğu dizini tespit et
current_dir = Path(os.getcwd())

# src dizinini bul (prediction notebook'u için gerekli değil ama tutarlılık için)
src_paths = [
    # Lokal ortam: notebooks/ klasöründen bir üst dizindeki src
    current_dir.parent / 'src',
    # Lokal ortam: mevcut dizindeki src
    current_dir / 'src',
    # Kaggle/Colab ortamları
    Path('/kaggle/working/src'),
    Path('/content/src'),
    # Relative paths
    Path('../src'),
    Path('src'),
]

src_path = None
for path in src_paths:
    if path.exists() and path.is_dir():
        if str(path.parent) not in sys.path:
            sys.path.insert(0, str(path.parent))
        src_path = path
        print(f"✅ Found src directory at: {path}")
        break

# Spark NLP modüllerini import et
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.ml import Pipeline

print("✅ Modules imported")


## 2. Model Yükleme


In [ ]:
# Eğitilmiş modeli yükle
model_path = "models/trained/custom_ner_model"

if not Path(model_path).exists():
    print(f"❌ Model not found: {model_path}")
    print("Please run 2_model_training.ipynb first to train the model.")
else:
    print(f"Loading trained model from {model_path}...")
    custom_model = MedicalNerModel.load(model_path)
    print("✅ Model loaded successfully")
    
    # Embeddings yükle (model için gerekli)
    print("Loading embeddings...")
    embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
        .setInputCols(["sentence", "token"])\
        .setOutputCol("embeddings")
    print("✅ Embeddings loaded")


## 3. Prediction Pipeline Oluşturma


In [ ]:
# Prediction pipeline oluştur
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")\
    .setCleanupMode("shrink")

sentence_detector = SentenceDetector()\
    .setInputCols(["document"])\
    .setOutputCol("sentence")\
    .setExplodeSentences(True)

tokenizer = Tokenizer()\
    .setInputCols(["sentence"])\
    .setOutputCol("token")

# Model'i pipeline'a ekle
custom_model.setInputCols(["sentence", "token", "embeddings"])\
    .setOutputCol("ner")

# NER sonuçlarını chunk'lara dönüştür
ner_converter = NerConverter()\
    .setInputCols(["document", "token", "ner"])\
    .setOutputCol("ner_chunks")

# Pipeline oluştur
prediction_pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    embeddings,
    custom_model,
    ner_converter
])

print("✅ Prediction pipeline created")


## 4. Tahmin Yapma


In [ ]:
# Örnek metinler (kendi metinlerinizi buraya ekleyebilirsiniz)
sample_texts = [
    "The patient was prescribed Aspirin 100mg twice daily for pain management.",
    "Dr. Smith recommended Metformin 500mg for diabetes treatment.",
    "The patient has a history of hypertension and is currently taking Lisinopril 10mg."
]

# DataFrame oluştur
from pyspark.sql.types import StringType
from pyspark.sql import Row

text_df = spark.createDataFrame([Row(text=text) for text in sample_texts])
print(f"Created DataFrame with {text_df.count()} texts")
text_df.show(truncate=100)


In [ ]:
# Pipeline'ı fit et ve tahmin yap
print("Running prediction pipeline...")
pipeline_model = prediction_pipeline.fit(text_df)
predictions = pipeline_model.transform(text_df)

print("✅ Predictions completed")
predictions.select("text", "ner_chunks").show(truncate=100)


## 5. Sonuçları Görselleştirme


In [ ]:
# Entity'leri detaylı göster
from pyspark.sql import functions as F

# Entity'leri extract et
entities_df = predictions.select(
    "text",
    F.explode(F.arrays_zip(
        predictions["ner_chunks"].result,
        predictions["ner_chunks"].begin,
        predictions["ner_chunks"].end,
        predictions["ner_chunks"].metadata
    )).alias("entity")
).select(
    "text",
    F.expr("entity['0']").alias("chunk"),
    F.expr("entity['1']").alias("begin"),
    F.expr("entity['2']").alias("end"),
    F.expr("entity['3']['entity']").alias("entity_type")
)

print("Extracted entities:")
entities_df.show(truncate=False)


In [ ]:
# Spark NLP Display ile görselleştir
try:
    from sparknlp_display import NerVisualizer
    
    # İlk metni görselleştir
    first_text = sample_texts[0]
    first_prediction = predictions.filter(predictions.text == first_text).first()
    
    if first_prediction:
        print(f"\nVisualization for: {first_text}\n")
        NerVisualizer().display(
            first_prediction,
            label_col='ner',
            document_col='document'
        )
except ImportError:
    print("⚠️ spark-nlp-display not available for visualization")
    print("Entity results are shown above")
except Exception as e:
    print(f"⚠️ Visualization error: {e}")
    print("Entity results are shown above")


## Özet

✅ **Prediction tamamlandı!**

**Yapılanlar:**
- Eğitilmiş custom NER modeli yüklendi
- Yeni metinler üzerinde entity tahminleri yapıldı
- Sonuçlar görselleştirildi

**Not:** Kendi metinlerinizi kullanmak için `sample_texts` listesini güncelleyebilir veya CSV dosyasından yükleyebilirsiniz.
